In [ ]:
# ==========================================================
# 05 - Error Analysis
# ==========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score
)

print("=" * 60)
print("Loading Data")
print("=" * 60)

df = pd.read_parquet("../data/processed/featured_taxi_data.parquet")

drop_columns = [
    "trip_duration_minutes",
    "tpep_dropoff_datetime",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "tolls_amount",
    "extra",
    "airport_fee",
    "Airport_fee",
    "congestion_surcharge",
    "improvement_surcharge"
]

X = df.drop(columns=drop_columns)
y = df["trip_duration_minutes"]

print("Loading Champion Model...")

model = joblib.load("../models/champion_model.joblib")

predictions = model.predict(X)

results = pd.DataFrame({
    "Actual": y,
    "Predicted": predictions
})

results["Error"] = results["Actual"] - results["Predicted"]
results["Absolute Error"] = results["Error"].abs()

print("\nEvaluation Metrics")

mae = mean_absolute_error(y, predictions)
rmse = np.sqrt(mean_squared_error(y, predictions))
medae = median_absolute_error(y, predictions)
r2 = r2_score(y, predictions)

print(f"MAE               : {mae:.4f}")
print(f"RMSE              : {rmse:.4f}")
print(f"Median AE         : {medae:.4f}")
print(f"R2 Score          : {r2:.4f}")

print("\n90 Percentile Error")
print(results["Absolute Error"].quantile(0.90))

within2 = (
    results["Absolute Error"] <= 2
).mean() * 100

within5 = (
    results["Absolute Error"] <= 5
).mean() * 100

within10 = (
    results["Absolute Error"] <= 10
).mean() * 100

print(f"Within 2 Minutes  : {within2:.2f}%")
print(f"Within 5 Minutes  : {within5:.2f}%")
print(f"Within 10 Minutes : {within10:.2f}%")

# ==========================================================
# Actual vs Predicted
# ==========================================================

plt.figure(figsize=(7,7))

plt.scatter(
    results["Actual"],
    results["Predicted"],
    alpha=0.3
)

plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted")

plt.show()

# ==========================================================
# Residual Distribution
# ==========================================================

plt.figure(figsize=(8,5))

plt.hist(
    results["Error"],
    bins=60
)

plt.title("Residual Distribution")

plt.show()

# ==========================================================
# Residual Plot
# ==========================================================

plt.figure(figsize=(8,5))

plt.scatter(
    results["Predicted"],
    results["Error"],
    alpha=0.3
)

plt.axhline(
    y=0,
    color="red"
)

plt.xlabel("Predicted")

plt.ylabel("Residual")

plt.title("Residual Plot")

plt.show()

# ==========================================================
# Error Distribution
# ==========================================================

plt.figure(figsize=(8,5))

plt.hist(
    results["Absolute Error"],
    bins=50
)

plt.title("Absolute Error Distribution")

plt.show()

# ==========================================================
# Segment Analysis
# ==========================================================

df["Absolute Error"] = results["Absolute Error"]

segment = df.groupby("is_peak_hour")["Absolute Error"].mean()

print("\nPeak Hour Error")

print(segment)

weekend = df.groupby("is_weekend")["Absolute Error"].mean()

print("\nWeekend Error")

print(weekend)

airport = df.groupby("airport_pickup")["Absolute Error"].mean()

print("\nAirport Pickup Error")

print(airport)

long_trip = df[df["trip_duration_minutes"] > 30]

print("\nLong Trip MAE")

print(long_trip["Absolute Error"].mean())

print("\nTop 10 Worst Predictions")

print(
    results.sort_values(
        "Absolute Error",
        ascending=False
    ).head(10)
)

results.to_csv(
    "../reports/error_analysis.csv",
    index=False
)

print("\nError Analysis Saved Successfully")